<a href="https://colab.research.google.com/github/cyruslayo/nba_bot/blob/main/notebooks/nba_market_models_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NBA Market Models — Colab Training Notebook

Trains the `spread`, `total`, and `first_half` classification models used by `nba_bot`.

This notebook mirrors the repository training pipeline in `nba_bot/train.py` and uses the same feature builders from `nba_bot/features.py`.

Artifacts are saved to Google Drive for later use by `nba-bot-scan`.

In [ ]:
# Cell 1 — Setup
!pip install "numpy<2" "pandas<3" "nba-on-court>=0.2.0,<1.0" xgboost scikit-learn joblib tqdm nba_api -q

import os

if not os.path.exists('/content/nba_bot'):
    !git clone https://github.com/cyruslayo/nba_bot.git /content/nba_bot
else:
    !git -C /content/nba_bot pull

!pip install -e /content/nba_bot -q

print('✅ Setup complete')

In [ ]:
# Cell 2 — Mount Drive
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_OUTPUT = '/content/drive/MyDrive/nba_bot/'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print('Output dir:', DRIVE_OUTPUT)

In [ ]:
# Cell 3 — Training config
SEASONS = [2021, 2022, 2023, 2024]
MARKET_TYPES = ['spread', 'total', 'first_half']
USE_ADVANCED_FEATURES = True
SAMPLE_GAMES_PER_SEASON = None
TEST_SIZE = 0.2
RANDOM_STATE = 42

print('Seasons:', SEASONS)
print('Market types:', MARKET_TYPES)
print('Advanced features:', USE_ADVANCED_FEATURES)
print('Sample games/season:', SAMPLE_GAMES_PER_SEASON)

In [ ]:
# Cell 4 — Load play-by-play data
import os
import pandas as pd
import nba_on_court as noc

def fix_columns(df):
    if df is None:
        return None
    df.columns = [c.upper() for c in df.columns]
    if 'GAME_ID' not in df.columns:
        possible_id_cols = [c for c in df.columns if 'NBASTATS' in c or 'GAMEID' in c or 'GAME_ID' in c]
        if possible_id_cols:
            df = df.rename(columns={possible_id_cols[0]: 'GAME_ID'})
    return df

def load_data_safe(seasons, data_type):
    try:
        df = noc.load_nba_data(seasons=seasons, data=data_type)
        if df is not None and len(df) > 0:
            return fix_columns(df)
    except Exception as e:
        print(f'Library load failed for {data_type}: {e}')

    dfs = []
    for season in seasons:
        fpath = f'/content/{data_type}_{season}.tar.xz'
        if os.path.exists(fpath):
            raw = pd.read_csv(fpath, compression='xz')
            dfs.append(fix_columns(raw))
    return pd.concat(dfs, ignore_index=True) if dfs else None

df_nbastats = load_data_safe(SEASONS, 'nbastats')
if df_nbastats is None or df_nbastats.empty:
    raise ValueError('Could not load nbastats data')

df_pbpstats = None
if USE_ADVANCED_FEATURES:
    df_pbpstats = load_data_safe(SEASONS, 'pbpstats')

print(f'nbastats rows: {len(df_nbastats):,}')
print(f'nbastats games: {df_nbastats["GAME_ID"].nunique():,}')
if df_pbpstats is not None:
    print(f'pbpstats rows: {len(df_pbpstats):,}')
    print(f'pbpstats games: {df_pbpstats["GAME_ID"].nunique():,}')

if SAMPLE_GAMES_PER_SEASON:
    game_ids = df_nbastats['GAME_ID'].dropna().astype(str)
    season_prefix = game_ids.str[:4]
    sampled_ids = []
    for prefix in sorted(season_prefix.unique()):
        ids = sorted(df_nbastats.loc[season_prefix == prefix, 'GAME_ID'].dropna().unique().tolist())
        sampled_ids.extend(ids[:SAMPLE_GAMES_PER_SEASON])
    df_nbastats = df_nbastats[df_nbastats['GAME_ID'].isin(sampled_ids)].copy()
    if df_pbpstats is not None:
        df_pbpstats = df_pbpstats[df_pbpstats['GAME_ID'].isin(sampled_ids)].copy()
    print(f'Sampled nbastats rows: {len(df_nbastats):,}')
    print(f'Sampled games: {df_nbastats["GAME_ID"].nunique():,}')

print('✅ Data load complete')

In [ ]:
# Cell 5 — Load team ratings for T2 features
import importlib.util
import sys

player_ratings = {}
if USE_ADVANCED_FEATURES:
    !sed -i "s/per_mode_simple/per_mode_detailed/g" /content/nba_bot/nba_bot/team_stats_cache.py

    config_path = '/content/nba_bot/nba_bot/config.py'
    config_spec = importlib.util.spec_from_file_location('nba_bot.config', config_path)
    config_mod = importlib.util.module_from_spec(config_spec)
    sys.modules['nba_bot.config'] = config_mod
    config_spec.loader.exec_module(config_mod)

    module_file = '/content/nba_bot/nba_bot/team_stats_cache.py'
    spec = importlib.util.spec_from_file_location('nba_bot.team_stats_cache', module_file)
    tsc_module = importlib.util.module_from_spec(spec)
    sys.modules['nba_bot.team_stats_cache'] = tsc_module
    spec.loader.exec_module(tsc_module)

    refresh_team_stats = tsc_module.refresh_team_stats
    player_ratings = refresh_team_stats(output_path='/content/team_stats.json')
    print(f'Loaded ratings for {len(player_ratings)} teams')
else:
    print('Skipping team ratings')

In [ ]:
# Cell 6 — Build feature datasets
import importlib.util
import sys

features_path = '/content/nba_bot/nba_bot/features.py'
spec = importlib.util.spec_from_file_location('nba_bot.features', features_path)
features_module = importlib.util.module_from_spec(spec)
sys.modules['nba_bot.features'] = features_module
spec.loader.exec_module(features_module)

build_spread_rows = features_module.build_spread_rows
build_total_rows = features_module.build_total_rows
build_first_half_rows = features_module.build_first_half_rows

from nba_bot.config import FEATURES_SPREAD, FEATURES_TOTAL, FEATURES_FIRST_HALF

datasets = {}

if 'spread' in MARKET_TYPES:
    df_spread = build_spread_rows(
        df_nbastats=df_nbastats,
        df_pbpstats=df_pbpstats,
        player_ratings=player_ratings,
    )
    datasets['spread'] = {
        'df': df_spread,
        'feature_cols': FEATURES_SPREAD,
        'target_col': 'covered_spread',
        'model_name': 'xgb_spread_t2.pkl',
        'feature_name': 'feature_cols_spread.pkl',
    }
    print('spread shape:', df_spread.shape)

if 'total' in MARKET_TYPES:
    df_total = build_total_rows(
        df_nbastats=df_nbastats,
        df_pbpstats=df_pbpstats,
        player_ratings=player_ratings,
    )
    datasets['total'] = {
        'df': df_total,
        'feature_cols': FEATURES_TOTAL,
        'target_col': 'went_over',
        'model_name': 'xgb_total_t2.pkl',
        'feature_name': 'feature_cols_total.pkl',
    }
    print('total shape:', df_total.shape)

if 'first_half' in MARKET_TYPES:
    df_first_half = build_first_half_rows(
        df_nbastats=df_nbastats,
        df_pbpstats=df_pbpstats,
        player_ratings=player_ratings,
    )
    datasets['first_half'] = {
        'df': df_first_half,
        'feature_cols': FEATURES_FIRST_HALF,
        'target_col': 'home_win',
        'model_name': 'xgb_first_half_t2.pkl',
        'feature_name': 'feature_cols_first_half.pkl',
    }
    print('first_half shape:', df_first_half.shape)

In [ ]:
# Cell 7 — Train and save all selected models
import joblib
import pandas as pd
import xgboost as xgb
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split

results = {}

for market_type, bundle in datasets.items():
    print('=' * 70)
    print('Training:', market_type)

    df_market = bundle['df'].copy()
    feature_cols = bundle['feature_cols']
    target_col = bundle['target_col']

    df_market = df_market[df_market['time_remaining'] < 2870].copy()
    df_market = df_market.dropna(subset=feature_cols + [target_col])

    print('Rows after filter:', len(df_market))
    print('Target mean:', round(float(df_market[target_col].mean()), 4))

    X = df_market[feature_cols].values
    y = df_market[target_col].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )

    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        verbosity=0,
    )
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    probs = model.predict_proba(X_test)[:, 1]
    metrics = {
        'log_loss': round(float(log_loss(y_test, probs)), 4),
        'brier_score': round(float(brier_score_loss(y_test, probs)), 4),
        'auc_roc': round(float(roc_auc_score(y_test, probs)), 4),
    }
    print('Metrics:', metrics)

    model_path = os.path.join(DRIVE_OUTPUT, bundle['model_name'])
    feature_path = os.path.join(DRIVE_OUTPUT, bundle['feature_name'])
    joblib.dump(model, model_path)
    joblib.dump(feature_cols, feature_path)

    results[market_type] = {
        'metrics': metrics,
        'model_path': model_path,
        'feature_path': feature_path,
    }

    print('Saved model:', model_path)
    print('Saved features:', feature_path)

print('✅ Training complete for:', ', '.join(results.keys()))

In [ ]:
# Cell 8 — Export summary
for market_type, info in results.items():
    print('=' * 70)
    print(market_type)
    print('metrics:', info['metrics'])
    print('model:', info['model_path'])
    print('features:', info['feature_path'])

print('')
print('Set these locally after downloading the artifacts:')
print('  export NBA_BOT_SPREAD_MODEL_PATH=/path/to/xgb_spread_t2.pkl')
print('  export NBA_BOT_TOTAL_MODEL_PATH=/path/to/xgb_total_t2.pkl')
print('  export NBA_BOT_FIRST_HALF_MODEL_PATH=/path/to/xgb_first_half_t2.pkl')
print('  export NBA_BOT_ENABLE_SPREAD_TRADING=true')
print('  export NBA_BOT_ENABLE_TOTAL_TRADING=true')
print('  export NBA_BOT_ENABLE_FIRST_HALF_TRADING=true')